# Visual representation diagnostics — full five-seed run
**Run all cells**, from `notebooks/medium_world`, **after** `representation_checkpoints.ipynb`.
Five seeds × two curricula × three checkpoints = **30 trained-checkpoint evaluations**, with random-encoder and shuffled-label controls. No pilot stage. Use the same question sets across seeds to enable paired comparisons; do not change the model or dataset.

This notebook evaluates expanded same-scene questions, oracle-validated counterfactuals, three frozen-feature probes, ViT-adapted Network Dissection, and signed-gradient-map spectral clustering (SpRAy-inspired, not original LRP). All result files are isolated from LR-reset outputs.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEEDS = (42, 43, 44, 45, 46)
SCHEDULES = ('prereq_mixed_pair_focus', 'mixed')
STEPS = (5400, 7200, 9000)
SPLITS = ('id_dev', 'ood_dev')
DEVICE = 'cuda'
DATA_DIR = PROJECT_ROOT / 'outputs/medium_world/dataset'
OUT_DIR = PROJECT_ROOT / 'outputs/medium_world/experiments/representation_diagnostics/five_seeds'
GRID_SCENES = {'id_dev': 512, 'ood_dev': 400}
COUNTERFACT_SCENES = {'id_dev': 256, 'ood_dev': 200}
PROBE_TRAIN_SCENES = 1200
PROBE_EVAL_SCENES = {'id_dev': 512, 'ood_dev': 400}
DISSECTION_CAL_SCENES = 384
DISSECTION_DEV_SCENES = 256
EXPLANATION_SCENES = {'id_dev': 160, 'ood_dev': 160}
SPRAY_CLUSTERS = 3

In [ ]:
import gc
import json
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.medium_visual_world.learning import MediumDataset, AttributeBindingViT, set_seed
from src.medium_visual_world.world import COLORS, SHAPES
from src.medium_visual_world.representation_probing import (
    distinct_scenes, same_scene_grid, predict_rows, consistency_report,
    counterfactual_rows, counterfactual_report,
    invariance_rows, invariance_report, extract_object_features,
    fit_probe, collect_dissection, make_explanation_questions,
    signed_explanations, spray_cluster, sha256_file,
    data_hash_audit, unique_question_rows,
)
from scipy.sparse.csgraph import connected_components
assert torch.cuda.is_available() or DEVICE == 'cpu'
device = torch.device(DEVICE)
torch.set_num_threads(min(4, torch.get_num_threads()))
train = MediumDataset(DATA_DIR / 'train.jsonl')
datasets = {name: MediumDataset(DATA_DIR / f'{name}.jsonl') for name in SPLITS}
hashes = data_hash_audit(DATA_DIR)
CODE_FILES = {
    'learning': 'src/medium_visual_world/learning.py',
    'scaffolded_spiral': 'src/medium_visual_world/scaffolded_spiral.py',
    'lr_policies': 'src/medium_visual_world/lr_policies.py',
    'render': 'src/medium_visual_world/render.py',
    'world': 'src/medium_visual_world/world.py',
    'prediction_tracking': 'src/medium_visual_world/prediction_tracking.py',
}
CODE_HASHES = {
    name: sha256_file(PROJECT_ROOT / path)
    for name, path in CODE_FILES.items()
}
manifest = pd.read_csv(OUT_DIR / 'checkpoint_manifest.csv')
assert len(manifest) == 30 and set(manifest.seed) == set(SEEDS) and (set(manifest.step) == set(STEPS))
assert set(manifest.schedule) == set(SCHEDULES)
scene_sets = {'train': {r['scene_id'] for r in train.rows}}
scene_sets.update({split: {r['scene_id'] for r in data.rows} for split, data in datasets.items()})
for left, right in (('train', 'id_dev'), ('train', 'ood_dev'), ('id_dev', 'ood_dev')):
    assert scene_sets[left].isdisjoint(scene_sets[right]), f'Scene overlap {left}, {right}'
print('Verified', len(manifest), 'checkpoint paths and frozen scene-disjoint data')

In [ ]:
def load_diagnostic_checkpoint(path):
    # Older checkpoints saved torch.__version__ as the TorchVersion str subclass.
    # Allowlist precisely that type while keeping restricted weights-only loading.
    with torch.serialization.safe_globals([torch.torch_version.TorchVersion]):
        return torch.load(path, map_location='cpu', weights_only=True)

def load_checkpoint(schedule, seed, step):
    record = manifest[(manifest.schedule == schedule) & (manifest.seed == seed) & (manifest.step == step)]
    assert len(record) == 1
    path = Path(record.iloc[0]['path'])
    assert path.is_file() and sha256_file(path) == record.iloc[0]['checkpoint_sha256']
    saved = load_diagnostic_checkpoint(path)
    assert saved['schedule'] == schedule and saved['seed'] == seed and (saved['step'] == step)
    assert saved['dataset_hashes'] == hashes
    assert saved['code_hashes'] == CODE_HASHES, (
        'Training source code has changed since checkpoint generation'
    )
    model = AttributeBindingViT(**saved['model_config']).to(device)
    model.load_state_dict(saved['model'], strict=True)
    return (model.eval(), record.iloc[0]['checkpoint_sha256'])

def new_random_encoder(seed):
    set_seed(seed + 19293)
    return AttributeBindingViT(dim=48, patch_size=4, depth=1).to(device).eval()

def save_csv(frame, name):
    path = OUT_DIR / name
    frame.to_csv(path, index=False)
    return path
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            model, _ = load_checkpoint(schedule, seed, step)
            del model
print('All 30 model checkpoints load successfully')

## Build fixed diagnostic questions and oracle-validated edits
All ID-dev and OOD-dev scenes are used for same-scene logic. Counterfactuals use a deterministic *predeclared* scene subset, shared by all seeds; this is a full-design subset, not a one-seed pilot. The expanded question distribution is **not** the historical ID/OOD benchmark distribution. Identical Color and Shape questions occur repeatedly in the Cartesian logical matrix; they are deduplicated later for task accuracy.

In [ ]:
grids = {split: same_scene_grid(datasets[split].rows, GRID_SCENES[split], 29, split) for split in SPLITS}
edits = {split: counterfactual_rows(datasets[split].rows, COUNTERFACT_SCENES[split], 31, split) for split in SPLITS}
controls = {split: invariance_rows(edits[split]) for split in SPLITS}
for split in SPLITS:
    expected = GRID_SCENES[split] * len(COLORS) * len(SHAPES) * 4
    assert len(grids[split]) == expected
    assert len({r['scene_id'] for r in grids[split]}) == GRID_SCENES[split]
    assert edits[split] and len(edits[split]) % 2 == 0
    assert all((r['label'] in (0, 1) for r in grids[split] + edits[split] + controls[split]))
    print(split, 'grid:', len(grids[split]), 'changed pairs:', len(edits[split]) // 2, 'invariance control pairs:', len(controls[split]) // 2)

## A–B. Behavior and counterfactual tests
Every model is evaluated on identical question IDs. The Cartesian matrix is retained for logical constraints; use `question_key` to deduplicate repeated single-attribute questions for any task-level average.

In [ ]:
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            model, ckhash = load_checkpoint(schedule, seed, step)
            suffix = f'{schedule}_seed{seed}_step{step}'
            for split in SPLITS:
                meta = {
                    'schedule': schedule,
                    'seed': seed,
                    'step': step,
                    'checkpoint_sha256': ckhash
                }
                prediction = predict_rows(model, grids[split], device, record_head_features=True).assign(**meta)
                assert len(unique_question_rows(prediction)) == GRID_SCENES[split] * (5 + 4 + 20 + 20)
                save_csv(prediction, f'same_scene_{split}_{suffix}.csv')
                logic = consistency_report(prediction).assign(**meta)
                save_csv(logic, f'logical_consistency_{split}_{suffix}.csv')
                paired = counterfactual_report(predict_rows(model, edits[split], device)).assign(**meta)
                invariant = invariance_report(predict_rows(model, controls[split], device)).assign(**meta)
                save_csv(paired, f'counterfactual_{split}_{suffix}.csv')
                save_csv(invariant, f'invariance_{split}_{suffix}.csv')
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            print('Completed behavior', suffix, flush=True)

## C. Frozen per-object probes
Fit linear probes on fixed square regions centered on oracle object coordinates, with ViT fully frozen. Save **per-object predictions**, per-class metrics and random/shuffled-label controls. These oracle-centered readouts assess accessible information; they are **not** deployable object-detection performance.

In [ ]:
from src.medium_visual_world.world import COLORS, SHAPES
train_scenes = distinct_scenes(train.rows, PROBE_TRAIN_SCENES, 313)
probe_dev = {
    split: distinct_scenes(
        datasets[split].rows, PROBE_EVAL_SCENES[split], 314 + index
    )
    for index, split in enumerate(SPLITS)
}
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            original_model, ckhash = load_checkpoint(schedule, seed, step)
            suffix = f'{schedule}_seed{seed}_step{step}'
            result_rows = []
            prediction_rows = []
            for baseline in ('trained', 'random'):
                model = original_model if baseline == 'trained' else new_random_encoder(seed)
                info_train, features_train = extract_object_features(model, train_scenes, device, batch_size=32)
                extracted = {
                    split: extract_object_features(model, scenes, device, batch_size=32)
                    for split, scenes in probe_dev.items()
                }
                for stage in ('patch', 'encoder', 'fused'):
                    for task in ('shape', 'color'):
                        tr_y = info_train[task].to_numpy()
                        shuffled = np.random.default_rng(seed + 148 + (task == 'color')).permutation(tr_y)
                        for split, (info_eval, features_eval) in extracted.items():
                            y = info_eval[task].to_numpy()
                            for label_kind, target_train in [
                                ('true', tr_y), ('shuffled_train', shuffled)
                            ] if baseline == 'trained' else [('true', tr_y)]:
                                r = fit_probe(features_train[stage], target_train, features_eval[stage], y, seed)
                                result_rows.append({
                                    'schedule': schedule,
                                    'seed': seed,
                                    'step': step,
                                    'baseline': baseline,
                                    'probe_label': label_kind,
                                    'task': task,
                                    'stage': stage,
                                    'split': split,
                                    'n_train_objects': len(info_train),
                                    'n_eval_objects': len(info_eval),
                                    'accuracy': r['accuracy'],
                                    'macro_f1': r['macro_f1'],
                                    'checkpoint_sha256': ckhash
                                })
                                if label_kind == 'true':
                                    frame = info_eval.copy()
                                    frame['target'] = y
                                    frame['prediction'] = r['predictions']
                                    frame['correct'] = y == r['predictions']
                                    frame = frame.assign(schedule=schedule, seed=seed, step=step, baseline=baseline, task=task, stage=stage, split=split, checkpoint_sha256=ckhash)
                                    prediction_rows.append(frame)
                if baseline == 'random':
                    del model
            save_csv(pd.DataFrame(result_rows), f'probe_metrics_{suffix}.csv')
            save_csv(pd.concat(prediction_rows, ignore_index=True), f'probe_predictions_{suffix}.csv')
            del original_model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            print('Completed probes', suffix, flush=True)

## D. Network Dissection-style ViT channel alignment
Threshold each channel using **train-scene** activations (99.5th percentile). Compare renderer-derived Color/Shape masks with patch, encoder and fused stage maps on disjoint ID-dev scenes. Random networks are a descriptive control. Low single-channel overlap does not imply distributed information is absent.

In [ ]:
dissection_train = distinct_scenes(train.rows, DISSECTION_CAL_SCENES, 66)
dissection_dev = distinct_scenes(datasets['id_dev'].rows, DISSECTION_DEV_SCENES, 67)
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            original_model, ckhash = load_checkpoint(schedule, seed, step)
            for baseline in ('trained', 'random'):
                model = original_model if baseline == 'trained' else new_random_encoder(seed)
                rows = collect_dissection(model, dissection_train, dissection_dev, device, quantile=(0.99, 0.995))
                rows = rows.assign(schedule=schedule, seed=seed, step=step, baseline=baseline, checkpoint_sha256=ckhash)
                save_csv(rows, f'dissection_{schedule}_seed{seed}_step{step}_{baseline}.csv')
                if baseline == 'random':
                    del model
            del original_model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            print('Completed dissection', schedule, seed, step, flush=True)

## E. SpRAy-inspired spectral explanation clustering
We cluster **signed gradient × (image − background)** maps of the Yes−No logit margin, *not* raw images or transformer embeddings and *not* original LRP. Use fixed, scene-unique Binding questions; separately cluster within split and true label. Save graph spectra, per-map metadata, and signed 16×16 maps. If a graph is disconnected or maps have near-zero norm, retain explicit diagnostic flags.

In [ ]:
explanation_questions = {
    split: make_explanation_questions(datasets[split].rows, EXPLANATION_SCENES[split], 80 + index, split)
    for index, split in enumerate(SPLITS)
}
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            trained, ckhash = load_checkpoint(schedule, seed, step)
            for baseline in ('trained', 'random'):
                model = trained if baseline == 'trained' else new_random_encoder(seed)
                for split, questions in explanation_questions.items():
                    maps, meta = signed_explanations(model, questions, device, batch_size=12)
                    name = f'{split}_{schedule}_seed{seed}_step{step}_{baseline}'
                    np.savez_compressed(OUT_DIR / f'explanation_maps_{name}.npz', maps=maps)
                    meta = meta.assign(schedule=schedule, seed=seed, step=step, baseline=baseline, checkpoint_sha256=ckhash, explanation_method='signed_grad_x_image_minus_background',map_index=np.arange(len(meta)))
                    clusters = []
                    eigen_rows = []
                    for label in (0, 1):
                        indices = np.flatnonzero(meta.label.to_numpy() == label)
                        norm = np.linalg.norm(maps[indices].reshape(len(indices), -1), axis=1)
                        valid_indices = indices[norm >= 1e-10]
                        zero_indices = indices[norm < 1e-10]
                        for i in zero_indices:
                            clusters.append({
                                'map_index': int(i),
                                'cluster': np.nan,
                                'zero_map': True,
                                'label': label
                            })
                        if len(valid_indices) < max(24, SPRAY_CLUSTERS + 14):
                            raise RuntimeError(
                                f'Not enough valid explanation maps: {name}; '
                                f'label={label}, n={len(valid_indices)}'
                            )
                        ids, values, gaps, adjacency = spray_cluster(
                            maps[valid_indices],
                            n_clusters=SPRAY_CLUSTERS,
                            neighbors=min(12, len(valid_indices) - 2),
                            seed=seed
                        )
                        components = int(connected_components(adjacency, directed=False, return_labels=False))
                        for i, c in zip(valid_indices, ids):
                            clusters.append({
                                'map_index': int(i),
                                'cluster': int(c),
                                'zero_map': False,
                                'label': label
                            })
                        for index, value in enumerate(values):
                            eigen_rows.append({
                                'schedule': schedule,
                                'seed': seed,
                                'step': step,
                                'baseline': baseline,
                                'split': split,
                                'label': label,
                                'graph_components': components,
                                'checkpoint_sha256': ckhash,
                                'eigen_index': index,
                                'eigenvalue': float(value),
                                'gap_after': float(gaps[index]) if index < len(gaps) else np.nan,
                                'n_clustered_maps': len(valid_indices),
                                'n_zero_maps': len(zero_indices)
                            })
                    cluster_frame = meta.merge(pd.DataFrame(clusters), on=['map_index', 'label'], how='left', validate='one_to_one')
                    assert len(cluster_frame) == len(meta)
                    assert cluster_frame.cluster.notna().sum() + cluster_frame.zero_map.sum() == len(meta)
                    save_csv(cluster_frame, f'explanation_clusters_{name}.csv')
                    save_csv(pd.DataFrame(eigen_rows), f'explanation_eigenspectrum_{name}.csv')
                if baseline == 'random':
                    del model
            del trained
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            print('Completed explanations', schedule, seed, step, flush=True)

## Confirm complete inventory
Do not interpret or publish partial output as a completed five-seed experiment. The analysis notebook repeats this audit before reporting statistics.

In [ ]:
from pathlib import Path
required = []
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            suffix = f'{schedule}_seed{seed}_step{step}'
            PROBE_METRICS = OUT_DIR / f"probe_metrics_{suffix}.csv"
            PROBE_PREDICTIONS = OUT_DIR / f'probe_predictions_{suffix}.csv'
            required += [PROBE_METRICS, PROBE_PREDICTIONS]
            for split in SPLITS:
                required.extend((OUT_DIR / f'{prefix}_{split}_{suffix}.csv' for prefix in (
                    'same_scene',
                    'logical_consistency',
                    'counterfactual',
                    'invariance'
                )))
                for baseline in ('trained', 'random'):
                    name = f'{split}_{suffix}_{baseline}'
                    required += [
                        OUT_DIR / f'explanation_maps_{name}.npz',
                        OUT_DIR / f'explanation_clusters_{name}.csv',
                        OUT_DIR / f'explanation_eigenspectrum_{name}.csv'
                    ]
            for baseline in ('trained', 'random'):
                required.append(OUT_DIR / f'dissection_{suffix}_{baseline}.csv')
missing = [str(p) for p in required if not p.is_file()]
assert not missing, f'Missing {len(missing)} files: {missing[:8]}'
print('FULL EXPERIMENT COMPLETE:', len(SEEDS), 'seeds,', len(SCHEDULES), 'curricula,', len(STEPS), 'checkpoints,', len(required), 'verified output files.')

## Refined Shape measurement

For the spatially preserving, color-balanced Shape probe, run `representation_probe_refinement.ipynb` **after** this original diagnostic. Do not rerun this notebook solely for the refinement: the new notebook reuses the existing 30 frozen checkpoints and leaves these outputs unchanged.